# KnobNet Training
**순서**: 환경 설치 → Drive 마운트 → 데이터 압축 해제 → 학습

## 1. 환경 설치

In [ ]:
!pip install -q transformers soundfile torchaudio

## 2. GitHub 클론

In [ ]:
import os

GITHUB_REPO = "https://github.com/YOUR_USERNAME/KnobNet.git"  # <-- 수정
PROJECT_DIR = "/content/KnobNet"

if not os.path.exists(PROJECT_DIR):
    !git clone {GITHUB_REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print("현재 디렉터리:", os.getcwd())

## 3. Google Drive 마운트 & 데이터 압축 해제

Drive에 올린 zip 파일 경로를 `ZIP_FILES`에 입력하세요.  
예) `["MyDrive/KnobNet/data_part1.zip", "MyDrive/KnobNet/data_part2.zip"]`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

# Drive에 올린 첫 번째 분할 파일 경로 (수정)
# 7-Zip 분할: data.zip.001, .002, ...  → 첫 번째 파일만 입력
# 단일 zip:   data.zip                 → 해당 파일 입력
ZIP_FILES = [
    "/content/drive/MyDrive/KnobNet/data.zip.001",
    # 분할 zip이면 첫 번째 파일(.001)만 넣으면 7z가 나머지를 자동으로 읽음
    # 단일 zip 파일이 여러 개면 아래처럼 추가
    # "/content/drive/MyDrive/KnobNet/data2.zip",
]

DATA_DIR = Path(PROJECT_DIR) / "data"
DATA_DIR.mkdir(exist_ok=True)

# p7zip 설치 (분할 zip 지원)
!apt-get install -q p7zip-full

for zip_path in ZIP_FILES:
    zip_path = Path(zip_path)
    if not zip_path.exists():
        print(f"[SKIP] 파일 없음: {zip_path}")
        continue
    print(f"압축 해제 중: {zip_path.name} ...")
    !7z x "{zip_path}" -o"{DATA_DIR}" -y
    print("완료")

print("\n데이터 디렉터리 구조:")
!find {DATA_DIR} -maxdepth 3 -type d

## 4. 데이터 확인

In [ ]:
## 데이터 구조 진단 - 경로가 맞는지 확인 후 WET_DIR 수정
from pathlib import Path

DATA_DIR = Path(PROJECT_DIR) / "data"

print("=== data/ 하위 전체 디렉터리 ===")
!find {DATA_DIR} -type d | sort

print("\n=== samples.csv 위치 ===")
!find {DATA_DIR} -name "samples.csv" | sort

print("\n=== wav 파일 수 ===")
!find {DATA_DIR} -name "*.wav" | wc -l

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from dataset.loader import make_loaders

# dataset_root = PROJECT_DIR, wet_dir = "data/wet/black"
# (CSV의 input_file이 "data\clean-sequences\..." 형태이므로 PROJECT_DIR 기준)
train_loader, val_loader = make_loaders(
    dataset_root = PROJECT_DIR,
    wet_dir      = "data/wet/black",
    batch_size   = 16,
    val_split    = 0.2,
    num_workers  = 2,
)

from train.train import print_batch
inp, ref, knobs = next(iter(train_loader))
print_batch(inp, ref, knobs)

## 5. 모델 초기화

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

from model.model import KnobNet
from utils.config import KNOB_PARAMS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = KnobNet(num_knobs=len(KNOB_PARAMS), freeze_mert=True).to(device)
model.summary()

## 6. Phase 1: Head만 학습 (MERT frozen)

In [ ]:
from train.train import (
    make_optimizer, run_epoch, evaluate,
    evaluate_per_param, evaluate_accuracy,
    save_checkpoint, load_checkpoint,
    log_epoch, log_param_mae, log_accuracy,
)

# 체크포인트 경로 (Drive에 저장해서 세션 종료 후에도 유지)
CKPT_DIR = Path("/content/drive/MyDrive/KnobNet/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_P1 = CKPT_DIR / "phase1_best.pt"

PHASE1_EPOCHS = 30
criterion = nn.L1Loss()
optimizer = make_optimizer(model, lr=1e-3, phase=1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

# 이전 체크포인트가 있으면 재개
start_epoch = 1
best_val = float("inf")
if CKPT_P1.exists():
    meta = load_checkpoint(CKPT_P1, model, optimizer, scheduler)
    start_epoch = meta["epoch"] + 1
    best_val = meta["val_loss"]
    print(f"체크포인트 재개: epoch={meta['epoch']}  best_val={best_val:.4f}")

print(f"Phase 1 학습: epoch {start_epoch} ~ {PHASE1_EPOCHS}")
model.summary()

In [ ]:
for epoch in range(start_epoch, PHASE1_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer, criterion, device)
    val_loss   = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    improved = val_loss < best_val
    if improved:
        best_val = val_loss
        save_checkpoint(CKPT_P1, model, epoch, phase=1, val_loss=val_loss,
                        optimizer=optimizer, scheduler=scheduler)

    log_epoch(1, epoch, PHASE1_EPOCHS, train_loss, val_loss, improved)
    log_param_mae(evaluate_per_param(model, val_loader, device))
    log_accuracy(evaluate_accuracy(model, val_loader, device, tolerance=0.1), tolerance=0.1)

print("\nPhase 1 완료")

## 7. Phase 2: MERT fine-tuning

In [ ]:
CKPT_P2 = CKPT_DIR / "phase2_best.pt"
PHASE2_EPOCHS = 10

# Phase 1 best 체크포인트에서 weights 로드
load_checkpoint(CKPT_P1, model)

model.unfreeze_mert()
optimizer = make_optimizer(model, lr=1e-3, phase=2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

# Phase 2 체크포인트가 있으면 재개
start_epoch = 1
best_val = float("inf")
if CKPT_P2.exists():
    meta = load_checkpoint(CKPT_P2, model, optimizer, scheduler)
    start_epoch = meta["epoch"] + 1
    best_val = meta["val_loss"]
    print(f"Phase 2 재개: epoch={meta['epoch']}  best_val={best_val:.4f}")

print(f"Phase 2 학습: epoch {start_epoch} ~ {PHASE2_EPOCHS}")
model.summary()

In [ ]:
for epoch in range(start_epoch, PHASE2_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer, criterion, device)
    val_loss   = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    improved = val_loss < best_val
    if improved:
        best_val = val_loss
        save_checkpoint(CKPT_P2, model, epoch, phase=2, val_loss=val_loss,
                        optimizer=optimizer, scheduler=scheduler)

    log_epoch(2, epoch, PHASE2_EPOCHS, train_loss, val_loss, improved)
    log_param_mae(evaluate_per_param(model, val_loader, device))
    log_accuracy(evaluate_accuracy(model, val_loader, device, tolerance=0.1), tolerance=0.1)

print("\nPhase 2 완료")

## 8. 최종 모델 export

In [ ]:
EXPORT_PATH = CKPT_DIR / "knobnet_final.pt"

# Phase 2 best weights 로드 후 export
load_checkpoint(CKPT_P2, model)
model.export(EXPORT_PATH)

size_mb = EXPORT_PATH.stat().st_size / 1024 / 1024
print(f"저장 완료: {EXPORT_PATH}  ({size_mb:.1f} MB)")

## (선택) 예측 결과 확인

In [ ]:
from train.train import print_predictions

load_checkpoint(CKPT_P2, model)
print_predictions(model, val_loader, device, n=8)